# Retail Sales Forecasting: rebuilt chronological study

Project experiment using separately supplied Walmart-format CSVs. This notebook
calls the same audited study implementation as `python -m retail_study`; there is
no independent notebook metric or model-selection implementation.

The original positional-metric bug, random/KFold evaluation, holiday-flag
overwrites, positional submission assignment and unsupported historical scores
have been removed. Those defects do not establish deliberate falsification.

The protocol requires 143 labeled weekly dates and 39 external forecast weeks.
Development uses expanding windows 52→26 and 78→26 weeks; selection uses pooled
development WMAE. The final 104→39-week holdout is revealed only after selection
is frozen. External competition targets are hidden; local errors are never
presented as leaderboard scores. See README and the real-run report for evidence.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from IPython.display import display
from matplotlib import pyplot as plt
import seaborn as sns

from retail_study import load_inputs, study_splits, run_study

# Run Jupyter from the repository root. Edit settings here or set environment variables.
DATA_DIR = Path(os.environ.get("RETAIL_DATA_DIR", "data")).expanduser().resolve()
OUTPUT_DIR = Path(os.environ.get("RETAIL_OUTPUT_DIR", "outputs/study-run")).expanduser().resolve()
INPUT_KIND = os.environ.get("RETAIL_INPUT_KIND", "")  # explicitly choose real_public_mirror, real_official or synthetic
PROVENANCE_PATH = os.environ.get("RETAIL_PROVENANCE")  # required source-manifest JSON for real inputs
if INPUT_KIND not in {"real_public_mirror", "real_official", "synthetic"}:
    raise ValueError("Set INPUT_KIND explicitly; see README. Synthetic inputs verify code only.")
print(f"Input kind: {INPUT_KIND}\nInput: {DATA_DIR}\nFresh output directory: {OUTPUT_DIR}")

## Validate inputs and explore the earliest training prefix

Left joins preserve every panel row and reject duplicates, absent matches and
conflicting official holiday flags. All finite sales targets, including negative
and zero values, are retained. Integrity audits may inspect all input schemas;
the plots below use only the earliest 52 weeks, preceding both development blocks
and the final holdout. No feature choices or tuning follow final holdout errors.

In [ ]:
tables, train_detail, test_detail = load_inputs(DATA_DIR)
splits = study_splits(train_detail, test_detail)
eda_detail = splits["dev_1"][0].copy()
display(eda_detail.head())
display(eda_detail.isna().sum().rename("missing_early_training_values").to_frame())
display(pd.DataFrame([
    {"split": name, "training_rows": len(earlier), "validation_rows": len(later),
     "training_start": earlier.Date.min(), "training_end": earlier.Date.max(),
     "validation_start": later.Date.min(), "validation_end": later.Date.max()}
    for name, (earlier, later) in splits.items()
]))

The fixed learned-model features are Store, Dept, store Type, Size, ISO Week,
calendar Year, official IsHoliday and a separate IsEaster flag. The inherited
Easter mapping covers only 2010–2013 (ISO weeks 13,16,14,13); outside those years
it is false. It never changes the original official scoring flags. Marginal
correlation does not establish that omitted external covariates are useless.

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
weekly = eda_detail.groupby("Date").Weekly_Sales.agg(["mean", "median"])
weekly.plot(ax=axes[0])
axes[0].set_title("Earliest training prefix: weekly sales")
average = eda_detail.groupby("Store", as_index=False).Weekly_Sales.mean()
sns.barplot(data=average, x="Store", y="Weekly_Sales", color="steelblue", ax=axes[1])
axes[1].set_title("Earliest training prefix: mean sales per store")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
correlation = eda_detail.corr(numeric_only=True)
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
mask = np.triu(np.ones_like(correlation, dtype=bool))
sns.heatmap(correlation, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1, ax=axes[0])
axes[0].set_title("Earliest training prefix: numeric Pearson correlation")
sns.histplot(data=eda_detail, x="Weekly_Sales", bins=40, ax=axes[1])
axes[1].set_title("Earliest training prefix: all finite targets retained")
plt.tight_layout()
plt.show()
plt.close(fig)

## Run the predeclared study — this cell reveals the final holdout

The shared runner writes its audit/protocol before fitting, compares four fixed
candidates on two disjoint development blocks, and selects the lowest pooled
WMAE. Candidates: last observed sale; 52-week naive; weighted Random Forest
(56 trees); weighted histogram boosting (absolute error, 100 iterations).
Official holidays have weight 5; other rows weight 1. No random early stopping,
parameter search or holdout-driven redesign occurs. The candidate order in the
protocol breaks score ties within 1e-10.

Baselines consult only training history, falling back to the last series sale
and then the global training median. HGB's nominal categories are learned only
from each training prefix; unknowns become missing categories. RF uses numeric
thresholds for nominal IDs, a documented limitation, and squared-error fitting
despite absolute-error evaluation.

After `selection.json` freezes the winner, only that method and the two declared
baseline references are evaluated on the final 39 weeks. The selected method
is then reconstructed/refitted on **all 143 weeks** for external prediction.
A baseline winner remains the primary submission. Use a fresh output directory
on every run; existing evidence is not overwritten.

In [ ]:
study = run_study(DATA_DIR, OUTPUT_DIR, INPUT_KIND, PROVENANCE_PATH)
display(study["development_metrics"].round(4))
display(study["pooled_metrics"].round(4))
print("Frozen winner:", study["selection"]["model"])
display(study["final_metrics"].round(4))
print("Local final holdout errors; external leaderboard performance remains unknown.")

## Inspect the selected method's keyed external submission

Every finite prediction is matched by `Store_Dept_YYYY-MM-DD` to the sample
template's original ID order. Duplicate, missing or extra IDs fail visibly.
The persisted selected model is reloaded and checked for identical predictions.
The runner independently recomputes final metrics from the saved row-level CSV.

In [ ]:
submission = study["submission"]
display(submission.head())
print(f"Saved {len(submission)} keyed predictions to {OUTPUT_DIR / 'Submission.csv'}")
print(f"Audit, protocol, frozen selection, full-precision metrics and report: {OUTPUT_DIR}")

The report describes the short history, unequal development/final horizons,
single final holdout, correlated observations, represented holidays, sparse/new
series and external-period differences. Week-level spread is descriptive, not a
confidence interval based on independent panel rows. Synthetic fixtures verify
execution only. Public-mirror agreement does not authenticate official data.